# Two nonlinear FMI generators on a shared EMT three-phase bus

Two separately exported MATLAB/Simulink generator models retain the same machine, AVR and governor equations. Each export declares singleton instantiation; distinct model identities permit one instance of each. This does **not** claim duplicate instances of one singleton FMU are supported.

Both generators connect to one ABC bus and the same R/L/C network. Initial conditions come from a solved balanced operating point: 230 V phase RMS, 50 Hz, 8 kW per generator. Network C and load conductance are doubled; series R/L are halved, preserving the per-generator operating point. Physical coupling remains active when the first reference increases by 2% at exactly 0.05 s and the second decreases by 1% at exactly 0.075 s.

The independent MATLAB reference integrates one 26-state system using full implicit trapezoidal Newton solves. It does not call the FMUs or reproduce DPsim's network condensation. Segmented `ode15s` supplies a higher-accuracy cross-check. All 46 generator outputs and all six network channels are compared over the full trajectory, including both events. Bases are declared below; the 0.1% provisional max/RMS check remains explicit.


In [ ]:
from pathlib import Path
import subprocess, shutil, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

REPO = Path("/Volumes/SSDMatthias/development/dpsim")
CASE = REPO / "dpsim-fmi/examples/two_generators"
BUILD = REPO / "build-macos-check/fmi-validation/two_generators"
BUILD.mkdir(parents=True, exist_ok=True)
for asset in CASE.joinpath("matlab").iterdir():
    if asset.suffix in [".csv", ".json"]:
        shutil.copyfile(asset, BUILD / asset.name)
exe = REPO / "build-macos-check/dpsim-fmi/FMI_EMT_Ph3_TwoGenerators"
for name, h, tol, swap in [
    ("base", 5e-5, 1, False),
    ("fine", 2.5e-5, 1, False),
    ("tight", 5e-5, 0.1, False),
    ("swapped", 5e-5, 1, True),
]:
    folder = BUILD / name
    folder.mkdir(exist_ok=True)
    models = [
        CASE / "sources/first_generator.fmu",
        CASE / "sources/dpsim_sg_3ph_second.fmu",
    ]
    if swap:
        models.reverse()
    if (
        not (folder / "run.log").exists()
        or "completed:" not in (folder / "run.log").read_text()
    ):
        result = subprocess.run(
            [
                str(exe),
                *map(str, models),
                str(CASE / "sources/benchmark.json"),
                str(h),
                str(tol),
            ],
            cwd=folder,
            capture_output=True,
            text=True,
        )
        (folder / "run.log").write_text(result.stdout + result.stderr)
        assert result.returncode == 0, result.stderr

In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd

B = BUILD
C = json.loads((CASE / "sources/benchmark.json").read_text())
bases = C["bases"]


def base_for(name):
    if name.startswith("i_"):
        return 2 * bases["phase_current_A"]
    if name.startswith("v_pcc_"):
        return bases["phase_voltage_V"]
    s = name.split("_", 1)[1]
    if s in ["i_a", "i_b", "i_c"]:
        return bases["phase_current_A"]
    if s.startswith("p_"):
        return bases["power_W"]
    if s == "field_current_A":
        return bases["field_current_A"]
    if s == "psi_f_Wb":
        return bases["field_flux_Wb"]
    if s.startswith("psi_"):
        return bases["stator_flux_Wb"]
    return 1.0


def evaluate(case, reference):
    d = pd.read_csv(B / case / "logs/fmi_two_generators.csv", skipinitialspace=True)
    r = pd.read_csv(B / reference)
    assert len(d) == len(r) and np.max(abs(d.time - r.time)) < 1e-11
    pairs = {s: (d[s], r[s], base_for(s)) for s in r if s != "time" and s in d}
    for k, p in enumerate("abc"):
        pairs["v_pcc_" + p] = (
            d[f"pcc_voltage_{k}"],
            r["v_pcc_" + p],
            bases["phase_voltage_V"],
        )
        pairs["i_line_" + p] = (
            d[f"network_state_{k+3}"],
            r["i_line_" + p],
            2 * bases["phase_current_A"],
        )
    rows = []
    for signal, (a, b, base) in pairs.items():
        e = np.asarray(a) - np.asarray(b)
        peak = float(np.max(abs(e)))
        rms = float(np.sqrt(np.mean(e * e)))
        rows.append(
            dict(
                case=case,
                reference=reference,
                signal=signal,
                base=base,
                max_abs=peak,
                rms_abs=rms,
                max_normalized=peak / base,
                rms_normalized=rms / base,
                pass_0p1pct=peak / base < 0.001 and rms / base < 0.001,
            )
        )
    kcl = max(
        float(
            np.max(
                abs(
                    d[f"first_current_{k}"]
                    + d[f"second_current_{k}"]
                    + d[f"network_current_{k}"]
                )
            )
        )
        for k in range(3)
    )
    diagnostics = {
        "samples": len(d),
        "kcl_max_A": kcl,
        "first_residual_max": float(d.first_residual.max()),
        "second_residual_max": float(d.second_residual.max()),
        "first_reference_change_time": float(
            d.time.iloc[np.where(np.abs(d.first_vref - 1) > 0.001)[0][0]]
        ),
        "second_reference_change_time": float(
            d.time.iloc[np.where(np.abs(d.second_vref - 1) > 0.001)[0][0]]
        ),
    }
    assert (
        abs(diagnostics["first_reference_change_time"] - 0.05) < 1e-10
        and abs(diagnostics["second_reference_change_time"] - 0.075) < 1e-10
    )
    assert (
        np.max(abs(d.first_efd_pu - d.second_efd_pu)) > 1e-4
    ), "Controls did not separate"
    assert kcl < 1e-5 and max(d.first_residual.max(), d.second_residual.max()) <= 1
    pd.DataFrame(rows).to_csv(
        B / case / ("metrics_" + Path(reference).stem + ".csv"), index=False
    )
    (B / case / "diagnostics.json").write_text(json.dumps(diagnostics, indent=2) + "\n")
    return pd.DataFrame(rows), d, r, pairs, diagnostics

In [ ]:
results = {}
for case, reference in [
    ("base", "reference_trap_50us.csv"),
    ("fine", "reference_trap_25us.csv"),
    ("base", "reference_ode15s.csv"),
]:
    results[case, reference] = evaluate(case, reference)
metrics = pd.concat([v[0] for v in results.values()], ignore_index=True)
assert metrics.pass_0p1pct.all()
metrics.to_csv(BUILD / "metrics_all.csv", index=False)
display(pd.DataFrame(bases.items(), columns=["Engineering base", "Value"]))
display(
    metrics.groupby(["case", "reference"])[["max_normalized", "rms_normalized"]].max()
    * 100
)
display(metrics)
print(
    "Initial operating point:", json.loads((BUILD / "operating_point.json").read_text())
)
for item in results.values():
    print(item[-1])

In [ ]:
coarse = results["base", "reference_trap_50us.csv"][1]
for case in ["tight", "swapped"]:
    other = pd.read_csv(
        BUILD / case / "logs/fmi_two_generators.csv", skipinitialspace=True
    )
    columns = [s for s in coarse if "residual" not in s]
    difference = float(
        np.max(abs(coarse[columns].to_numpy() - other[columns].to_numpy()))
    )
    assert difference == 0
    print(case, "maximum logged difference:", difference)
# Respect the declared singleton restriction; do not modify its XML.
negative = subprocess.run(
    [
        str(exe),
        str(CASE / "sources/first_generator.fmu"),
        str(CASE / "sources/first_generator.fmu"),
        str(CASE / "sources/benchmark.json"),
    ],
    cwd=BUILD,
    capture_output=True,
    text=True,
)
assert negative.returncode != 0 and "only one instantiation" in negative.stderr
print(negative.stderr.strip())
a = pd.read_csv(BUILD / "reference_trap_50us.csv")
b = pd.read_csv(BUILD / "reference_trap_25us.csv").iloc[::2].reset_index(drop=True)
r = pd.read_csv(BUILD / "reference_ode15s.csv")
refinement = []
for signal in ["first_i_a", "second_i_a", "v_pcc_a", "first_efd_pu", "second_efd_pu"]:
    coarse_rms = float(np.sqrt(np.mean((a[signal] - r[signal]) ** 2)))
    fine_rms = float(np.sqrt(np.mean((b[signal] - r[signal]) ** 2)))
    refinement.append(
        dict(
            signal=signal,
            coarse_rms=coarse_rms,
            fine_rms=fine_rms,
            improvement=coarse_rms / fine_rms,
        )
    )
refinement = pd.DataFrame(refinement)
assert (refinement.improvement > 3.8).all()
refinement.to_csv(BUILD / "refinement.csv", index=False)
display(refinement)

In [ ]:
metrics, d, r, pairs, diagnostics = results["base", "reference_trap_50us.csv"]
fig, axes = plt.subplots(4, 1, figsize=(11, 11), sharex=True)
for ax, signals, unit in [
    (axes[0], ["first_i_a", "second_i_a"], "A"),
    (axes[1], ["first_efd_pu", "second_efd_pu"], "pu"),
    (axes[2], ["first_pm_pu", "second_pm_pu"], "pu"),
    (axes[3], ["v_pcc_a"], "V"),
]:
    for signal in signals:
        actual, expected, base = pairs[signal]
        ax.plot(d.time, expected, label=signal + " MATLAB", lw=1.3)
        ax.plot(d.time, actual, "--", label=signal + " DPsim", lw=0.8)
    ax.set_ylabel(unit)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
    for event in [0.05, 0.075]:
        ax.axvline(event, color="k", alpha=0.25, lw=0.7)
axes[-1].set_xlabel("Time [s]")
fig.tight_layout()
fig.savefig(BUILD / "comparison.png", dpi=160)
plt.show()
fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
for ax, signal in zip(axes, ["first_i_a", "second_i_a", "v_pcc_a"]):
    actual, expected, base = pairs[signal]
    ax.plot(d.time, 100 * (np.asarray(actual) - np.asarray(expected)) / base)
    ax.set_ylabel(signal + " [% base]")
    ax.grid(alpha=0.25)
axes[-1].set_xlabel("Time [s]")
fig.tight_layout()
fig.savefig(BUILD / "errors.png", dpi=160)
plt.show()

## Reproduction and scope

Build `FMI_EMT_Ph3_TwoGenerators` with the repository CMake configuration, then run this notebook. Each C++ invocation takes two FMU paths and `sources/benchmark.json`, with optional timestep and tolerance multiplier. Runs, logs, metrics and plots stay under `build-macos-check/fmi-validation/two_generators/`.

Saved MATLAB references and the independently assembled reference implementation are in `matlab/`. To regenerate, add this folder to the MATLAB path, select a fresh generated-output folder or remove the generated reference CSVs under build, and call `two_generator_reference`. It uses the retained generator model-equation functions in the neighboring generator case. The second export is made by saving the original generator Simulink model under a new model identity and exporting FMI 3.0 Model Exchange, with no changes to its equations or controls.

The operating point validates consistent nonzero initialization supplied by the model and matched external-network conditions. It is not an automatic initializer for arbitrary FMUs or a demonstrated warm restart of the switched converter. Rotor angle and abc network states rotate at steady AC operation; their time derivatives should not be forced to zero. The equilibrium residual concerns the stationary machine/controller states.
